# 🏭 Chest X-Ray — Produce All Outputs (Downstream Production Notebook)

This notebook supports both the original resumable downstream mode and a required
**patient-disjoint rebuild mode**. The latter regenerates splits, retrains every
affected model, and replaces all dependent results. Existing checkpoints from the
old image-level split must not be reused for final manuscript metrics.

- per-image predictions (`tiered_predictions.csv`, Ark+ and EfficientNet)
- A13 (tiered) and A14 (CheXpert zero-shot) evaluation JSONs (if missing)
- the genuine `ablation.json` (no fabrication), statistical tests, model comparison
- thesis figures + 5 analysis notebooks (from the real predictions)
- ONNX export + latency/carbon benchmark
- multi-batch FID-gated Stable Diffusion synthetic data (before Tier 2 training)

**Design principles:**
1. Every step has a `run` and a `force` (overwrite even if present) switch → `STEPS`.
2. `PATIENT_DISJOINT_REBUILD=True` forces a one-time clean scientific rerun. After
   that succeeds, set it to `False`; resumable skip-existing behavior is restored.
3. No cell crashes: each step is wrapped in try/except and a status report is printed at the end.
4. Saving to Drive **MERGEs into the single `xray_outputs/outputs` tree**; base model
   weights (.pth) are not re-uploaded, and no dated new zip copies are produced.

## 0. Configuration & Switches

In [ ]:
# ============================================================================
# ⚙️ CONFIGURATION — produce-all (downstream) pipeline
# ============================================================================
import importlib.util, os, shutil
from pathlib import Path
IS_COLAB = importlib.util.find_spec('google.colab') is not None
_cwd = Path.cwd().resolve()
_local_repo = _cwd if (_cwd / 'scripts').is_dir() else (_cwd.parent if (_cwd.parent / 'scripts').is_dir() else _cwd)
RUNTIME_ROOT = '/content' if IS_COLAB else os.environ.get('XRAY_RUNTIME_ROOT', str(_local_repo))
if IS_COLAB:
    shutil.rmtree('/content/sample_data', ignore_errors=True)

GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'rev/mdpi-peer-review'
PROJECT_ROOT     = os.environ.get('XRAY_PROJECT_ROOT', '/content/xray' if IS_COLAB else str(_local_repo))
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs'
USE_DRIVE        = IS_COLAB
KAGGLE_DATASET   = 'nih-chest-xrays/data'
DRY_RUN          = False
PATIENT_DISJOINT_REBUILD = True  # REQUIRED once; set False only after a successful clean rerun
PROTOCOL_VERSION = 'patient-disjoint-v1'

# --- Global switches -------------------------------------------------------
SKIP_EXISTING    = True    # skip a step if its outputs already exist
FORCE_ALL        = PATIENT_DISJOINT_REBUILD  # invalidates old-split checkpoints/results

# --- Compute knobs ---------------------------------------------------------
N_BOOT           = 2000    # bootstrap iterations for statistical tests
MC_PASSES        = 20      # MC-dropout passes for prediction export

# --- Drive save behaviour --------------------------------------------------
SAVE_TO_DRIVE      = USE_DRIVE  # local/server runs keep artifacts under PROJECT_ROOT
MAKE_ZIP_SNAPSHOT  = False   # also write ONE results zip (same path, overwritten — never dated copies)
BIG_FILE_MB        = 50      # files larger than this that were NOT produced now are treated as base
                             # weights and never re-uploaded to Drive

# --- Per-step switches: run = execute, force = overwrite even if outputs exist ---
# Toggle any single model/file independently here.
STEPS = {
    # NOTE: setup cells (Drive restore, NIH cache, CheXpert download)
    # are idempotent and self-skip via their own cache checks, so they are not listed
    # here. Every key below maps to exactly one produce() step in the cells.
    # --- Model training: SKIPS if weights already exist (restored from Drive).
    #     Set force=True (or FORCE_ALL, or delete the .pth) to retrain & overwrite. ---
    'synthetic_generation':     {'run': True,  'force': False},
    'train_tier1':              {'run': True,  'force': False},
    'train_tier2_efficientnet': {'run': True,  'force': False},
    'train_tier2_arkplus':      {'run': True,  'force': False},
    'ablation_A8':              {'run': True,  'force': False},
    'ablation_A9':              {'run': True,  'force': False},
    'ablation_A11':             {'run': True,  'force': False},
    'ablation_A12':             {'run': True,  'force': False},
    'ablation_A15':             {'run': True,  'force': False},
    'calibration_temperature':  {'run': True,  'force': False},
    'calibration_conformal':    {'run': True,  'force': False},
    'eval_A13_tiered':          {'run': True,  'force': False},
    'eval_A14_chexpert':        {'run': True,  'force': False},
    'eval_A14b_chexpert_large':  {'run': True,  'force': False},
    'revision_experiments':      {'run': True,  'force': False},
    # Full-table evaluation ablations A1-A12 + A15 (scripts/evaluate_ablation.py).
    'eval_A1':                  {'run': True,  'force': False},
    'eval_A2':                  {'run': True,  'force': False},
    'eval_A3':                  {'run': True,  'force': False},
    'eval_A4':                  {'run': True,  'force': False},
    'eval_A5':                  {'run': True,  'force': False},
    'eval_A5b':                 {'run': True,  'force': False},
    'eval_A6':                  {'run': True,  'force': False},
    'eval_A7':                  {'run': True,  'force': False},
    'eval_A8':                  {'run': True,  'force': False},
    'eval_A9':                  {'run': True,  'force': False},
    'eval_A10':                 {'run': True,  'force': False},
    'eval_A11':                 {'run': True,  'force': False},
    'eval_A12':                 {'run': True,  'force': False},
    'eval_A15':                 {'run': True,  'force': False},
    'predictions_arkplus':      {'run': True,  'force': False},
    'predictions_efficientnet': {'run': True,  'force': False},
    'figure_input_csvs':        {'run': True,  'force': False},
    'ablation_json':            {'run': True,  'force': False},
    'statistical_tests':        {'run': True,  'force': False},
    'compare_models':           {'run': True,  'force': False},
    'report_figures':           {'run': True,  'force': False},
    'nb_subgroup_analysis':     {'run': True,  'force': False},
    'nb_decision_curve':        {'run': True,  'force': False},
    'nb_error_analysis':        {'run': True,  'force': False},
    'nb_tier_disagreement':     {'run': True,  'force': False},
    'nb_synthetic_quality':     {'run': True,  'force': False},
    'onnx_tier1':               {'run': True,  'force': False},
    'onnx_tier2':               {'run': True,  'force': False},
    'benchmark_latency':        {'run': True,  'force': False},
}

# --- Training hyperparameters (DYNAMIC — edit here, nothing is hardcoded in code) ---
NUM_WORKERS    = 8       # DataLoader workers for training
DRY_RUN_EPOCHS = 1       # epochs per model when DRY_RUN=True
SYNTH_HP = dict(
    seed_scope='all',        # 'all' = isolated + comorbid Pneumothorax training seeds
    batch_size=200,          # real training seeds per FID-evaluated batch (n=600 avoids finite-sample FID bias)
    num_variations=3,        # synthetic variations per real seed
    strength=0.15,           # anatomy-preserving SD1.5 img2img denoising strength
    fid_threshold=50.0,      # Inception-v3 FID acceptance gate (<= 50.0)
    seed=42,                 # deterministic base seed
    resume=True,             # resume cleanly from last completed batch
    qualitative_grid_path='outputs/figures/synthetic_quality_grid.png',
)

TRAIN_HP = {
    'train_tier1':              dict(backbone='mobilenet_v2',    run_name='Tier1_MobileNetV2_Colab',
                                     epochs=50, batch_size=32, lr_backbone=1e-4, lr_head=1e-3, patience=7,
                                     use_synthetic=False),
    'train_tier2_efficientnet': dict(backbone='efficientnet_b4', run_name='Tier2_EfficientNetB4_Colab',
                                     epochs=50, batch_size=16, lr_backbone=5e-5, lr_head=5e-4, patience=7,
                                     use_synthetic=True),
    'train_tier2_arkplus':      dict(backbone='ark_plus',        run_name='Tier2_ArkPlus_Colab',
                                     epochs=40, batch_size=12, lr_backbone=2e-5, lr_head=2e-4, patience=8,
                                     use_synthetic=True),
}

# --- Data thresholds ---
MIN_NIH_IMAGES = 50000   # below this, NIH images are (re)downloaded

# --- Ablation training experiments: step -> (experiment id, run_name/folder) ---
ABLATIONS = {
    'ablation_A8':  ('A8',  'A8_NoSynthetic'),
    'ablation_A9':  ('A9',  'A9_NoAugmentation'),
    'ablation_A11': ('A11', 'A11_ArkPlus_Only_NoMCTTA'),
    'ablation_A12': ('A12', 'A12_ArkPlus_Only_MC_TTA'),
    'ablation_A15': ('A15', 'A15_Mixup_Cutmix'),
}

# --- Downstream evaluation ablations (full A1-A15 table): step -> (id, run_name) ---
# Each loads REAL trained weights and computes genuine NIH-test metrics into
# outputs/results/<run_name>.json (the honest source build_ablation_json.py reads).
# A13/A14 have their own dedicated cell above and are not repeated here.
# --- Ablation EVAL knobs (DYNAMIC - tune to your GPU; see scripts/evaluate_ablation.py) ---
# The old evaluator ran batch_size=1 with MC/TTA as ~200 sequential forwards per
# image (GPU ~idle, 6h on T4). The new one VECTORISES MC+TTA into big batched
# forwards. micro_batch is the VRAM knob: raise it on A100 (e.g. 1024), lower on
# T4 (e.g. 256). Set mc_passes/tta_passes lower to trade a little accuracy for speed.
EVAL_HP = dict(
    batch_size=16,      # images per dataset batch (outer loop)
    micro_batch=512,    # max images per GPU forward (VRAM knob: A100~1024, T4~256)
    num_workers=NUM_WORKERS,
    amp=True,           # fp16 autocast on CUDA (~2x faster, half the VRAM)
    mc_passes=None,     # None = each row's spec (20); e.g. 10 for speed
    tta_passes=None,    # None = each row's spec (10); e.g. 5 for speed
)

EVAL_ABLATIONS = {
    'eval_A1':  ('A1',  'A1_Tier1_Only'),
    'eval_A2':  ('A2',  'A2_Tier2_EfficientNet_Only'),
    'eval_A3':  ('A3',  'A3_Tiered_Static'),
    'eval_A4':  ('A4',  'A4_Tiered_Dynamic'),
    'eval_A5':  ('A5',  'A5_Tiered_MC_Only'),
    'eval_A5b':  ('A5b', 'A5b_Tiered_TTA_Only'),
    'eval_A6':  ('A6',  'A6_Tiered_MC_TTA'),
    'eval_A7':  ('A7',  'A7_Tiered_Conformal'),
    'eval_A8':  ('A8',  'A8_NoSynthetic'),
    'eval_A9':  ('A9',  'A9_NoAugmentation'),
    'eval_A10': ('A10', 'A10_Always_Tier2'),
    'eval_A11': ('A11', 'A11_ArkPlus_Only_NoMCTTA'),
    'eval_A12': ('A12', 'A12_ArkPlus_Only_MC_TTA'),
    'eval_A15': ('A15', 'A15_Mixup_Cutmix'),
}
# --- Analysis notebooks: step -> notebook stem ---
NB_STEPS = {
    'nb_subgroup_analysis':  'subgroup_analysis',
    'nb_decision_curve':     'decision_curve_analysis',
    'nb_error_analysis':     'error_analysis',
    'nb_tier_disagreement':  'tier_disagreement',
    'nb_synthetic_quality':  'synthetic_quality',
}

# --- SINGLE SOURCE OF TRUTH: each step's primary output marker (skip-existing + data gating) ---
MARKERS = {
    'train_tier1':              'outputs/models/best_tier1_mobilenet.pth',
    'train_tier2_efficientnet': 'outputs/models/best_tier2_efficientnet.pth',
    'train_tier2_arkplus':      'outputs/models/best_tier2_ark_plus.pth',
    'calibration_temperature':  'outputs/results/temperature.pt',
    'calibration_conformal':    'outputs/results/q_hat.pt',
    'eval_A13_tiered':          'outputs/results/A13_Tiered_ArkPlus.json',
    'eval_A14_chexpert':        'outputs/results/A14_CheXpert_ZeroShot.json',
    'eval_A14b_chexpert_large':  'outputs/results/A14b_CheXpert_Large_ZeroShot.json',
    'revision_experiments':      'outputs/results/order_sensitivity.json',
    'predictions_arkplus':      'outputs/results/tiered_predictions.csv',
    'predictions_efficientnet': 'outputs/results/tiered_predictions_efficientnet.csv',
    'ablation_json':            'outputs/results/ablation.json',
    'statistical_tests':        'outputs/results/statistical_comparison.csv',
    'compare_models':           'outputs/results/model_comparison_bootstrap.csv',
    'report_figures':           'thesis/figures/threshold_sweep_4panel.png',
    'onnx_tier1':               'outputs/models/tier1_mobilenet.onnx',
    'onnx_tier2':               'outputs/models/tier2_efficientnet.onnx',
    'benchmark_latency':        'outputs/results/benchmark_latency.txt',
    'synthetic_generation':     'data/synthetic/generation_complete.json',
}
MARKERS.update({s: f'outputs/models/{rn}/best_model.pth' for s, (_id, rn) in ABLATIONS.items()})
MARKERS.update({s: f'outputs/notebooks/{stem}.ipynb' for s, stem in NB_STEPS.items()})
MARKERS.update({s: f'outputs/results/{rn}.json' for s, (_id, rn) in EVAL_ABLATIONS.items()})

# --- Conditional data download: which steps consume which dataset ---
NIH_CONSUMERS = (
    ['synthetic_generation', 'train_tier1', 'train_tier2_efficientnet', 'train_tier2_arkplus']
    + list(ABLATIONS) + list(EVAL_ABLATIONS)
    + ['calibration_temperature', 'calibration_conformal', 'eval_A13_tiered',
       'predictions_arkplus', 'predictions_efficientnet']
)
CHEXPERT_CONSUMERS = ['eval_A14_chexpert', 'eval_A14b_chexpert_large']

# --- Directories mirrored to/from Drive (so figures + notebooks sync too, not just outputs) ---
SYNC_DIRS = ['outputs', 'thesis/figures']

print('✅ Config loaded.')
print(f'  • SKIP_EXISTING={SKIP_EXISTING}  FORCE_ALL={FORCE_ALL}')
print(f'  • Drive merge → {DRIVE_OUTPUT_DIR}  (sync dirs: {SYNC_DIRS}, zip snapshot={MAKE_ZIP_SNAPSHOT})')
print(f'  • {sum(1 for s in STEPS.values() if s["run"])}/{len(STEPS)} steps enabled')

## 1. Environment Diagnostics & GPU

In [ ]:
import os
import sys

import torch

print(f'🐍 Python version: {sys.version.split()[0]}')
print(f'🔥 PyTorch version: {torch.__version__}')
print(f'⚡ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   🚀 GPU:   {device_name}')
    print(f'   💾 VRAM:  {total_vram:.1f} GB')
    if total_vram > 30:
        print('   ✨ Excellent! Detected high-end A100 GPU. Optimum batch loading and tensor parallelism are active.')
else:
    print('   ⚠️  No GPU detected. Under Runtime > Change runtime type, select a GPU accelerator.')

## 2. Mount Google Drive

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    print('📂 Mounting Google Drive...')
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive mounted! Outputs will sync to: {DRIVE_OUTPUT_DIR}')
else:
    print('⏭️ Google Drive mount skipped.')

## 3. Clone Repo & Install Dependencies

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

# Capture Colab's preinstalled CUDA/ABI packages BEFORE invoking pip so
# PyTorch, torchvision, and NumPy are never downgraded or replaced.
_ABI_PACKAGES = ('torch', 'torchvision', 'numpy')
_ABI_PINS = []
for _pkg in _ABI_PACKAGES:
    try:
        _ABI_PINS.append(f'{_pkg}=={importlib.metadata.version(_pkg)}')
    except importlib.metadata.PackageNotFoundError:
        pass

_CONSTRAINTS_PATH = Path('/tmp/colab-abi-constraints.txt')
_CONSTRAINTS_PATH.write_text('\n'.join(_ABI_PINS) + ('\n' if _ABI_PINS else ''), encoding='utf-8')
if _ABI_PINS:
    print('🔒 Preserving Colab preinstalled ABI:', ', '.join(_ABI_PINS))

if not os.path.exists(PROJECT_ROOT):
    print(f'📥 Cloning repository from {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...')
    subprocess.check_call(['git', 'clone', '-b', GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT])
else:
    print(f'📂 Repository already exists at {PROJECT_ROOT}. Syncing branch {GITHUB_BRANCH}...')
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(['git', 'fetch', 'origin', GITHUB_BRANCH])
    subprocess.check_call(['git', 'checkout', GITHUB_BRANCH])
    subprocess.check_call(['git', 'pull', '--ff-only', 'origin', GITHUB_BRANCH])

os.chdir(PROJECT_ROOT)
for _p in (os.path.join(PROJECT_ROOT, 'src'), PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print('\n📦 Installing lean Colab Python 3.13 dependencies (requirements-colab.txt)...')
_pip_cmd = [sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt']
if _ABI_PINS:
    _pip_cmd.extend(['-c', str(_CONSTRAINTS_PATH)])
subprocess.check_call(_pip_cmd)

print('📦 Installing local tcrp package (editable, no-deps)...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-build-isolation', '--no-deps'
])

# Verify the scientific stack + tcrp import cleanly in a fresh process.
_check = subprocess.run(
    [
        sys.executable,
        '-c',
        (
            'import numpy, pandas, torch, torchvision, tcrp; '
            'print(f"numpy={numpy.__version__} pandas={pandas.__version__} '
            'torch={torch.__version__} torchvision={torchvision.__version__} '
            'cuda={torch.cuda.is_available()}")'
        ),
    ],
    capture_output=True,
    text=True,
)
if _check.returncode != 0:
    raise RuntimeError(
        'Fresh-process import verification failed:\n' + (_check.stderr or _check.stdout)
    )
print(f'✅ Stack verified in fresh process: {_check.stdout.strip()}')
print('✅ Codebase synced and all Colab dependencies configured successfully!')


## 3.5 Environment Doctor (torch/torchvision import health — sympy repair)

In [ ]:
# ----------------------------------------------------------------------------
# Environment doctor: make sure torch + torchvision import cleanly BEFORE any
# in-kernel use. A dependency resolve (onnxscript/requirements) can leave sympy
# half-installed, which surfaces deep in torchvision -> torch._dynamo -> torch.fx
# as: "AttributeError: module 'sympy' has no attribute 'core'". We auto-repair it
# here (in a fresh process), so the later in-kernel imports work.
# ----------------------------------------------------------------------------
import subprocess, sys

def _torch_ok():
    r = subprocess.run(
        [sys.executable, '-c', 'import torch, torchvision; print(torch.__version__, torchvision.__version__)'],
        capture_output=True, text=True)
    tail = (r.stderr or r.stdout).strip().splitlines()
    return r.returncode == 0, (tail[-1] if tail else '')

_ok, _msg = _torch_ok()
if _ok:
    print(f'✅ torch / torchvision import cleanly: {_msg}')
else:
    print(f'⚠️ torch/torchvision import failed: {_msg[:200]}')
    print('🔧 Repairing sympy (force-reinstall) + re-pinning torch-compatible deps...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'sympy>=1.13.3'])
    _ok, _msg = _torch_ok()
    if _ok:
        print(f'✅ Repaired (fresh process): {_msg}')
        print('   ⚠️ If a later cell STILL errors on torch/torchvision, do '
              'Runtime > Restart session then Run all (downloads + weights are '
              'preserved; skip-existing resumes where it left off).')
    else:
        print(f'❌ Still failing: {_msg[:200]}')
        print('   → Runtime > Restart session, then Run all. Data is preserved; skip-existing resumes.')


## 4. Production Engine (skip/force, merge-save infrastructure)

In [ ]:
# ============================================================================
# 🛠️ PRODUCTION ENGINE: idempotent, never-crashing step runner
# ============================================================================
import os, sys, subprocess, traceback, time

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PRODUCED = set()    # absolute paths created/refreshed in THIS session
RUN_REPORT = []     # (step, status, detail)

# NOTE: named _step_cfg (not _cfg) on purpose — a reused setup cell uses `_cfg`
# as a loop variable, which would otherwise shadow this helper in the shared
# notebook namespace and make produce() raise "'dict' object is not callable".
def _step_cfg(name):
    c = STEPS.get(name, {'run': True, 'force': False})
    return bool(c.get('run', True)), (bool(c.get('force', False)) or FORCE_ALL)

def _exist_all(paths):
    return bool(paths) and all(os.path.exists(p) for p in paths)

def _will_run(name):
    '''Predict whether step `name` will actually execute (drives conditional data download).'''
    run, force = _step_cfg(name)   # force already folds in FORCE_ALL
    if not run:
        return False
    if force or not SKIP_EXISTING:
        return True
    marker = MARKERS.get(name)
    return not (marker and os.path.exists(marker))

def _log(step, status, detail=''):
    icon = {'DONE':'✅','SKIPPED':'⏭️','DISABLED':'🚫','BLOCKED':'⛔','PARTIAL':'⚠️','ERROR':'❌'}.get(status, '•')
    print(f'{icon} [{status}] {step}' + (f' — {detail}' if detail else ''))
    RUN_REPORT.append((step, status, str(detail)))

def run_script(args, env=None):
    e = dict(os.environ)
    if env:
        e.update(env)
    print('   $ ' + ' '.join(str(a) for a in args))
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    for line in proc.stdout:
        print('   | ' + line, end='')
    return proc.wait()

def sh(args, env=None):
    rc = run_script(args, env)
    if rc != 0:
        raise RuntimeError(f'command failed (exit {rc}): ' + ' '.join(str(a) for a in args))

def produce(name, outputs=None, fn=None, inputs=None, always=False):
    '''Run step `name` defensively.

    - run=False            -> DISABLED
    - missing inputs       -> BLOCKED (no throw)
    - outputs exist + skip -> SKIPPED (unless force / always / FORCE_ALL)
    - else run fn()        -> DONE / PARTIAL / ERROR (never raises out of this cell)
    Output paths of a successful run are recorded in PRODUCED (drives Drive save).
    '''
    outputs = outputs or []
    inputs = inputs or []
    run, force = _step_cfg(name)
    if not run:
        _log(name, 'DISABLED'); return
    missing = [p for p in inputs if not os.path.exists(p)]
    if missing:
        _log(name, 'BLOCKED', f'missing inputs: {missing}'); return
    if SKIP_EXISTING and not force and not always and _exist_all(outputs):
        _log(name, 'SKIPPED', 'outputs present'); return
    t0 = time.time()
    try:
        if fn is not None:
            fn()
        if outputs and not _exist_all(outputs):
            miss = [p for p in outputs if not os.path.exists(p)]
            _log(name, 'PARTIAL', f'ran but missing: {miss}'); return
        for p in outputs:
            PRODUCED.add(os.path.abspath(p))
        _log(name, 'DONE', f'{time.time() - t0:.1f}s')
    except Exception as exc:
        traceback.print_exc()
        _log(name, 'ERROR', repr(exc))

print('✅ Engine ready: produce(name, outputs, fn, inputs, always).')


## 5. Kaggle Credentials (for NIH/CheXpert download)

In [ ]:
import os, json
from pathlib import Path

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(exist_ok=True)
KAGGLE_JSON_PATH = KAGGLE_CONFIG_DIR / 'kaggle.json'

# @markdown ### 🌐 Option A: Copy-Paste Kaggle API Credentials (Recommended)
# @markdown Copy-paste your Kaggle username and API key directly from your Kaggle settings:
KAGGLE_USERNAME = "" # @param {type:"string"}
KAGGLE_KEY = "" # @param {type:"string"}

if KAGGLE_USERNAME and KAGGLE_KEY:
    with open(KAGGLE_JSON_PATH, 'w') as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    KAGGLE_JSON_PATH.chmod(0o600)
    print(f"✅ Kaggle credentials saved from form fields! User: {KAGGLE_USERNAME}")
elif KAGGLE_JSON_PATH.exists():
    print(f"✅ Kaggle credentials already present at: {KAGGLE_JSON_PATH}")
else:
    # @markdown ---
    # @markdown ### 📤 Option B: Direct File Upload
    # @markdown If form fields are empty, upload your downloaded `kaggle.json` file here:
    try:
        from google.colab import files
        print("📤 Upload your kaggle.json:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            Path('kaggle.json').rename(KAGGLE_JSON_PATH)
            KAGGLE_JSON_PATH.chmod(0o600)
            print("✅ Kaggle credentials uploaded and installed successfully!")
        else:
            print("⏭️ Upload skipped or incomplete.")
    except Exception as e:
        print(f"⏭️ Direct upload skipped (non-interactive environment): {e}")

## 6. Restore Models & Prior Outputs (single Drive tree — no zip needed)

In [ ]:
# ----------------------------------------------------------------------------
# Restore models + prior outputs from Drive — SINGLE-TREE aware (no zip needed).
# This is the exact inverse of the Drive-save cell: it mirrors the whole
# xray_outputs/outputs tree back, PRESERVING structure (per-ablation subfolders
# <run>/best_model.pth and results/*.json), so A8-A15 + A13/A14 skip-existing
# detection works off a plain Drive folder. Zip / loose-weight fallbacks remain.
# ----------------------------------------------------------------------------
import glob, os, shutil, sys, zipfile

print('🔍 Restoring models + prior outputs from Drive (single-tree aware)...')
models_dir = os.path.join(PROJECT_ROOT, 'outputs', 'models')
outputs_dir = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(models_dir, exist_ok=True)
restored = 0

# (1) PRIMARY: mirror every Drive sync dir back, structure preserved (figures, models,
#     notebooks, results, per-ablation subfolders — everything).
_any_tree = False
if USE_DRIVE:
    for _rel in SYNC_DIRS:
        src = os.path.join(DRIVE_OUTPUT_DIR, _rel)
        if not os.path.isdir(src):
            continue
        _any_tree = True
        dst = os.path.join(PROJECT_ROOT, _rel)
        print(f'📂 Mirroring {src} -> {dst} (subfolders preserved)...')
        for root, _, files in os.walk(src):
            for fn in files:
                sp = os.path.join(root, fn)
                dp = os.path.join(dst, os.path.relpath(sp, src))
                os.makedirs(os.path.dirname(dp), exist_ok=True)
                if not os.path.exists(dp):
                    shutil.copy2(sp, dp); restored += 1
    print(f'   ✅ {restored} files restored from the Drive sync tree(s)')
if not _any_tree:
    print('   (no Drive sync tree — trying zip / loose-weight fallbacks)')

# (2) ZIP fallback (legacy archives in /content or Drive).
zips = glob.glob('/content/*.zip')
if USE_DRIVE and os.path.isdir('/content/drive/MyDrive'):
    zips += glob.glob('/content/drive/MyDrive/*.zip') + glob.glob(os.path.join(DRIVE_OUTPUT_DIR, '*.zip'))
for zp in dict.fromkeys(os.path.realpath(z) for z in zips if os.path.exists(z)):
    try:
        with zipfile.ZipFile(zp) as zf:
            names = zf.namelist()
            target = PROJECT_ROOT if any(n.lstrip('./').startswith('outputs/') for n in names) else outputs_dir
            zf.extractall(target)
        print(f'   📦 extracted {os.path.basename(zp)} -> {target}')
    except Exception as e:
        print(f'   ⚠️ zip skip {os.path.basename(zp)}: {e}')

# (3) Loose scattered core weights fallback — ONLY top-level best_*.pth.
#     Never flatten the per-ablation best_model.pth (shared basename -> collisions;
#     step (1) already restored them WITH their subfolder).
if USE_DRIVE and os.path.isdir('/content/drive/MyDrive'):
    for pat in ('/content/drive/MyDrive/xray_outputs/**/best_*.pth',
                '/content/drive/MyDrive/xray_notebooks/**/best_*.pth',
                '/content/drive/MyDrive/best_*.pth'):
        for m in glob.glob(pat, recursive=True):
            if os.path.basename(m) == 'best_model.pth':
                continue
            dst = os.path.join(models_dir, os.path.basename(m))
            if not os.path.exists(dst):
                shutil.copy2(m, dst); restored += 1

# (4) Filename alignment: best_tier2_arkplus.pth <-> best_tier2_ark_plus.pth.
_a = os.path.join(models_dir, 'best_tier2_arkplus.pth')
_b = os.path.join(models_dir, 'best_tier2_ark_plus.pth')
if os.path.exists(_a) and not os.path.exists(_b): shutil.copy2(_a, _b)
if os.path.exists(_b) and not os.path.exists(_a): shutil.copy2(_b, _a)

# (5) De-nest core weights if a restore landed them under outputs/models/models/...
for rf in ('best_tier1_mobilenet.pth', 'best_tier2_efficientnet.pth',
           'best_tier2_ark_plus.pth', 'best_tier2_arkplus.pth'):
    top = os.path.join(models_dir, rf)
    if not os.path.exists(top):
        nested = glob.glob(os.path.join(models_dir, '**', rf), recursive=True)
        if nested:
            shutil.copy2(nested[0], top)

# (6) Status report.
print('\n⚙️ Core checkpoints:')
for rf in ('best_tier1_mobilenet.pth', 'best_tier2_efficientnet.pth', 'best_tier2_ark_plus.pth'):
    p = os.path.join(models_dir, rf)
    print(f'  {"✅" if os.path.exists(p) else "❌"} {rf}'
          + (f' ({os.path.getsize(p) / 1e6:.1f} MB)' if os.path.exists(p) else '  [MISSING]'))

print('\n🧭 Ablation / eval completion (feeds skip-existing):')
try:
    for _p in (os.path.join(PROJECT_ROOT, 'src'), PROJECT_ROOT):
        if _p not in sys.path:
            sys.path.insert(0, _p)
    os.chdir(PROJECT_ROOT)
    from tcrp.training import AblationRunner
    from scripts.run_ablation import completion_marker, is_complete
    _runner = AblationRunner()
    for _e in ('A8', 'A9', 'A11', 'A12', 'A13', 'A14', 'A15'):
        _c = _runner.ablation_configs[_e]
        _ok = is_complete(_c)
        print(f'  {"✅ done  → SKIP" if _ok else "⬜ missing → RUN "}  {_e:<4} {completion_marker(_c)}')
except Exception as _err:
    print(f'  ⚠️ could not compute completion status: {_err}')


## 7. NIH Image Cache + Splits (CONDITIONAL — downloads only if needed)

In [ ]:
# ----------------------------------------------------------------------------
# Conditional NIH cache: download ONLY if a step that needs NIH images will run.
# (All models present + no retrain requested -> no NIH download at all.)
# ----------------------------------------------------------------------------
import glob, os, shutil, subprocess, sys
from pathlib import Path

DRIVE_DATA   = '/content/drive/MyDrive/nih_data'
COLAB_DATA   = os.environ.get('XRAY_NIH_DATA', os.path.join(RUNTIME_ROOT, 'nih_data'))
if not IS_COLAB and os.path.isdir(os.path.join(PROJECT_ROOT, 'data', 'raw', 'images')):
    COLAB_DATA = os.path.join(PROJECT_ROOT, 'data', 'raw')
MERGED_DIR   = f'{COLAB_DATA}/images'
DRIVE_MERGED = f'{DRIVE_DATA}/images'
CSV_NAME     = 'Data_Entry_2017.csv'

def _count(d):
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]) if os.path.isdir(d) else 0

_run_steps = [s for s in NIH_CONSUMERS if _will_run(s)]
need_nih = bool(_run_steps)

if not need_nih:
    print('⏭️ NIH download SKIPPED — no enabled step needs NIH images')
    print('   (every NIH-consuming step is already present or disabled).')
else:
    print(f'📦 NIH images needed by: {_run_steps}')
    os.makedirs(COLAB_DATA, exist_ok=True)
    n_local = _count(MERGED_DIR)
    if n_local >= MIN_NIH_IMAGES:
        print(f'✅ NIH images already present: {n_local}.')
    elif _count(DRIVE_MERGED) >= MIN_NIH_IMAGES:
        print('📂 Restoring cached NIH images from Drive...')
        os.makedirs(MERGED_DIR, exist_ok=True)
        subprocess.check_call(['cp', '-r', f'{DRIVE_MERGED}/.', MERGED_DIR])
    else:
        kj = Path.home() / '.kaggle' / 'kaggle.json'
        has_auth = kj.exists() or bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
        if not has_auth:
            print('⚠️ Kaggle credentials missing (set KAGGLE_USERNAME and KAGGLE_KEY env vars or kaggle.json). NIH NOT downloaded.')
        else:
            print(f'🌐 Downloading NIH ({KAGGLE_DATASET}) from Kaggle (one-time)...')
            os.environ['KAGGLE_CONFIG_DIR'] = str(kj.parent)
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
                                   '-p', COLAB_DATA, '--unzip'])
            os.makedirs(MERGED_DIR, exist_ok=True)
            for i in range(1, 13):
                sub = os.path.join(COLAB_DATA, f'images_{i:03d}', 'images')
                if os.path.isdir(sub):
                    for f in os.listdir(sub):
                        shutil.move(os.path.join(sub, f), os.path.join(MERGED_DIR, f))
                    shutil.rmtree(os.path.join(COLAB_DATA, f'images_{i:03d}'), ignore_errors=True)
            print(f'   ✅ merged {_count(MERGED_DIR)} images.')
            if (USE_DRIVE and os.path.isdir('/content/drive/MyDrive')
                    and _count(MERGED_DIR) >= MIN_NIH_IMAGES and not os.path.isdir(DRIVE_MERGED)):
                print('💾 Caching images to Drive for next time...')
                os.makedirs(DRIVE_MERGED, exist_ok=True)
                subprocess.call(f'cp -r {MERGED_DIR}/. {DRIVE_MERGED}/', shell=True)

    # Metadata CSV + splits (only meaningful once images exist).
    csv_dst = os.path.join(PROJECT_ROOT, 'data', 'raw', CSV_NAME)
    os.makedirs(os.path.dirname(csv_dst), exist_ok=True)
    for cand in (os.path.join(COLAB_DATA, CSV_NAME), os.path.join(DRIVE_DATA, CSV_NAME)):
        if os.path.exists(cand):
            shutil.copy2(cand, csv_dst); break
    os.makedirs(os.path.join(PROJECT_ROOT, 'data', 'processed'), exist_ok=True)
    with open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'image_dir.txt'), 'w') as f:
        f.write(MERGED_DIR)
    if _count(MERGED_DIR) > 0:
        os.chdir(PROJECT_ROOT)
        subprocess.run([sys.executable, 'scripts/preprocess.py', '--image-dir', MERGED_DIR], check=False)
        # Integrity gate: every patient must belong to exactly one partition.
        import hashlib, json, pandas as pd
        _split_names = ['train', 'val', 'calibration', 'test']
        _split_frames = {}
        _patient_sets = {}
        for _sn in _split_names:
            _sp = f'data/processed/{_sn}.csv'
            if not os.path.exists(_sp):
                raise RuntimeError(f'missing required split: {_sp}')
            _df = pd.read_csv(_sp)
            _split_frames[_sn] = _df
            _patient_sets[_sn] = set(_df['Image Index'].astype(str).str.split('_').str[0])
        _overlaps = {}
        for _i, _a in enumerate(sorted(_split_names)):
            for _b in sorted(_split_names)[_i + 1:]:
                _overlap = _patient_sets[_a] & _patient_sets[_b]
                _overlaps[f'{_a}__{_b}'] = len(_overlap)
                if _overlap:
                    raise RuntimeError(f'PATIENT LEAKAGE: {_a}/{_b} share {len(_overlap)} patients')
        _manifest = {
            'protocol': PROTOCOL_VERSION,
            'seed': 42,
            'splits': {},
            'pairwise_patient_overlaps': _overlaps,
        }
        for _sn, _df in _split_frames.items():
            _raw = open(f'data/processed/{_sn}.csv', 'rb').read()
            _manifest['splits'][_sn] = {
                'rows': len(_df), 'patients': len(_patient_sets[_sn]),
                'positives': int(_df['Label'].sum()), 'sha256': hashlib.sha256(_raw).hexdigest()}
        os.makedirs('outputs/provenance', exist_ok=True)
        with open('outputs/provenance/split_manifest.json', 'w') as _mf:
            json.dump(_manifest, _mf, indent=2)
        print('   ✅ patient-disjoint splits + calibration verified; manifest written.')


## 7.5 Stable Diffusion Synthetic Augmentation Batches (Pre-Training)

In [ ]:
# ----------------------------------------------------------------------------
# Multi-batch FID-gated Stable Diffusion synthetic generation (runs BEFORE
# Tier 2 training so synthetic-enabled models consume accepted patient-disjoint
# synthetics while A8/A9 remain strictly real-only).
# ----------------------------------------------------------------------------
def _synth():
    cmd = [
        sys.executable, '-u', 'scripts/generate_synthetics.py',
        '--train-csv', 'data/processed/train.csv',
        '--output-dir', 'data/synthetic',
        '--seed-scope', str(SYNTH_HP['seed_scope']),
        '--batch-size', str(2 if DRY_RUN else SYNTH_HP['batch_size']),
        '--num-variations', str(2 if DRY_RUN else SYNTH_HP['num_variations']),
        '--strength', str(SYNTH_HP['strength']),
        '--fid-threshold', str(SYNTH_HP['fid_threshold']),
        '--seed', str(SYNTH_HP['seed']),
        '--qualitative-grid-path', str(SYNTH_HP['qualitative_grid_path']),
        '--resume' if SYNTH_HP.get('resume', True) else '--no-resume',
    ]
    if DRY_RUN:
        cmd.extend(['--max-seeds', '4'])
    sh(cmd)

produce(
    'synthetic_generation',
    outputs=['data/synthetic/synthetic_metadata.csv', 'data/synthetic/fid_log.csv', 'data/synthetic/generation_complete.json'],
    fn=_synth,
    inputs=['data/processed/train.csv'],
)

## 8. Model Training — Base (Tier1 / Tier2-Eff / Tier2-Ark) + Ablation (A8/A9/A11/A12/A15)

> If weights already exist (restored from Drive) these steps are **SKIPPED**. To retrain and overwrite a specific model, set `STEPS['train_...']['force']=True` (or `FORCE_ALL=True`). `DRY_RUN=True` → 1 epoch per model.

In [ ]:
# ----------------------------------------------------------------------------
# MODEL TRAINING (ported from xray_colab_training_auto.ipynb).
# Each model SKIPS if its weights already exist (restored from Drive). To retrain
# and overwrite, set STEPS['<step>']['force'] = True (or FORCE_ALL, or delete the
# .pth). DRY_RUN=True -> 1 epoch per model (fast end-to-end test).
# Training runs in an isolated subprocess (clean GPU memory + exact hyperparameters).
# ----------------------------------------------------------------------------
def _train(step, legacy):
    # All hyperparameters come from TRAIN_HP / NUM_WORKERS / DRY_RUN_EPOCHS (config cell).
    hp = TRAIN_HP[step]
    epochs = DRY_RUN_EPOCHS if DRY_RUN else hp['epochs']
    lines = [
        f"import sys, os, shutil; sys.path.insert(0, os.path.join({PROJECT_ROOT!r}, 'src')); sys.path.insert(0, {PROJECT_ROOT!r})",
        "from tcrp.training import TrainingConfig, train_model",
        f"cfg = TrainingConfig(backbone={hp['backbone']!r}, run_name={hp['run_name']!r}, "
        f"epochs={epochs}, batch_size={hp['batch_size']}, lr_backbone={hp['lr_backbone']}, "
        f"lr_head={hp['lr_head']}, early_stopping_patience={hp['patience']}, seed=42, "
        f"use_synthetic={hp.get('use_synthetic', False)!r}, num_workers={NUM_WORKERS})",
        f"res = train_model(cfg, dry_run={DRY_RUN})",
        "wp = res['model_weight_path']",
        f"[shutil.copy2(wp, p) for p in {legacy!r} if os.path.abspath(p) != os.path.abspath(wp)]",
        "print('TRAINED best_val_auc =', res.get('best_val_auc'))",
    ]
    sh([sys.executable, '-u', '-c', chr(10).join(lines)])

def _train_inputs(step):
    base = ['data/processed/train.csv']
    if TRAIN_HP[step].get('use_synthetic', False):
        base.append('data/synthetic/synthetic_metadata.csv')
    return base

produce('train_tier1', outputs=[MARKERS['train_tier1']],
        fn=lambda: _train('train_tier1', [MARKERS['train_tier1']]),
        inputs=_train_inputs('train_tier1'))

produce('train_tier2_efficientnet', outputs=[MARKERS['train_tier2_efficientnet']],
        fn=lambda: _train('train_tier2_efficientnet', [MARKERS['train_tier2_efficientnet']]),
        inputs=_train_inputs('train_tier2_efficientnet'))

def _train_ark():
    run_script([sys.executable, '-u', 'scripts/download_ark_plus.py'])  # warm Ark+/Swin cache
    _train('train_tier2_arkplus',
           ['outputs/models/best_tier2_arkplus.pth', MARKERS['train_tier2_arkplus']])
produce('train_tier2_arkplus', outputs=[MARKERS['train_tier2_arkplus']],
        fn=_train_ark, inputs=_train_inputs('train_tier2_arkplus'))

# Ablation trainings A8/A9/A11/A12/A15 (A13/A14 are evaluations, run later).
# A8 (no synthetic) and A9 (no augmentation) remain strictly real-only.
_REAL_ONLY_ABLATIONS = {'A8', 'A9'}
def _make_ablation(eid):
    def _run():
        cmd = [sys.executable, '-u', 'scripts/run_ablation.py', '--experiments', eid]
        if DRY_RUN:
            cmd.append('--dry-run')
        sh(cmd)
    return _run
for _step, (_eid, _rn) in ABLATIONS.items():
    _ab_inputs = (
        ['data/processed/train.csv']
        if _eid in _REAL_ONLY_ABLATIONS
        else ['data/processed/train.csv', 'data/synthetic/synthetic_metadata.csv']
    )
    produce(_step, outputs=[MARKERS[_step]], fn=_make_ablation(_eid),
            inputs=_ab_inputs)

## 8.5 Protocol Reset + Validation-Only Routing Threshold

In [ ]:
# In rebuild mode, calibration artifacts from the leaked split are invalid.
if PATIENT_DISJOINT_REBUILD:
    for _stale in ['outputs/results/q_hat.pt', 'outputs/results/temperature.pt',
                   'outputs/results/conformal_calibration_summary.json',
                   'outputs/models/tier1_mobilenet_threshold.json']:
        if os.path.exists(_stale):
            os.remove(_stale)
            print(f'♻️ removed stale protocol artifact: {_stale}')

# Select the static routing threshold on validation only (Youden J), never test.
def _fit_tier1_threshold():
    import json, numpy as np, torch
    from sklearn.metrics import roc_auc_score, roc_curve
    from torch.utils.data import DataLoader
    from tcrp.config import get_settings
    from tcrp.data.augmentation import ClassicalAugmentation
    from tcrp.data.datasets import NIHChestXrayDataset
    from tcrp.models import ModelFactory
    _device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _model = ModelFactory.create('mobilenet_v2', num_classes=2, pretrained=False).to(_device)
    _state = torch.load('outputs/models/best_tier1_mobilenet.pth', map_location=_device)
    _model.load_state_dict(_state.get('model_state_dict', _state)); _model.eval()
    _settings = get_settings()
    _tf = ClassicalAugmentation(image_size=_settings.data.image_size, is_training=False).pipeline
    _ds = NIHChestXrayDataset(csv_file='data/processed/val.csv', transform=_tf)
    _loader = DataLoader(_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    _ys, _ps = [], []
    with torch.no_grad():
        for _batch in _loader:
            _x, _y = _batch[0].to(_device), _batch[1]
            _p = torch.softmax(_model(_x), dim=1)[:, 1].cpu().numpy()
            _ys.extend(np.asarray(_y)); _ps.extend(_p)
    _ys, _ps = np.asarray(_ys), np.asarray(_ps)
    _fpr, _tpr, _thr = roc_curve(_ys, _ps)
    _idx = int(np.argmax(_tpr - _fpr))
    _payload = {'optimal_threshold': float(_thr[_idx]), 'val_auc': float(roc_auc_score(_ys, _ps)),
                'method': 'validation_youden_j', 'protocol': PROTOCOL_VERSION}
    os.makedirs('outputs/models', exist_ok=True)
    with open('outputs/models/tier1_mobilenet_threshold.json', 'w') as _f:
        json.dump(_payload, _f, indent=2)
    print('✅ validation-only routing threshold:', _payload)

produce('routing_threshold',
        outputs=['outputs/models/tier1_mobilenet_threshold.json'],
        fn=_fit_tier1_threshold,
        inputs=['outputs/models/best_tier1_mobilenet.pth', 'data/processed/val.csv'])


## 9. Temperature Calibration (Validation Split -> temperature.pt + ECE + reliability)

In [ ]:
# ----------------------------------------------------------------------------
# Post-hoc temperature scaling fitted STRICTLY on the Validation Split
# (data/processed/val.csv) BEFORE split-conformal calibration.
# Produces outputs/results/temperature.pt and the reliability figure.
# ----------------------------------------------------------------------------
def _calibrate_temperature():
    import math
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from tcrp.config import get_settings
    from tcrp.data.augmentation import ClassicalAugmentation
    from tcrp.data.datasets import NIHChestXrayDataset
    from tcrp.models import ModelFactory
    from tcrp.uncertainty.calibration import calibrate_temperature, compute_ece, plot_reliability_diagram

    weights_path = 'outputs/models/best_tier2_efficientnet.pth'
    backbone = 'efficientnet_b4'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ModelFactory.create(backbone, num_classes=2, pretrained=False)
    state = torch.load(weights_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    model.to(device).eval()

    settings = get_settings()
    tf = ClassicalAugmentation(image_size=settings.data.image_size, is_training=False).pipeline
    ds = NIHChestXrayDataset(csv_file='data/processed/val.csv', transform=tf)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

    _, raw_temperature = calibrate_temperature(model, loader, device)
    temperature = float(raw_temperature) if (math.isfinite(float(raw_temperature)) and float(raw_temperature) > 0.05) else 1.0

    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)):
                x, y = batch[0].to(device), batch[1]
            else:
                x, y = batch['image'].to(device), batch['label']
            p = torch.softmax(model(x) / float(temperature), dim=1)[:, 1].cpu().numpy()
            probs.append(p)
            labels.append(y.numpy() if hasattr(y, 'numpy') else np.array(y))
    probs = np.concatenate(probs)
    labels = np.concatenate(labels)
    ece = compute_ece(probs, labels, n_bins=15)

    os.makedirs('outputs/results', exist_ok=True)
    torch.save(
        {
            'temperature': float(temperature),
            'ece': float(ece),
            'split': 'val',
            'backbone': backbone,
            'weights_path': weights_path,
            'protocol': PROTOCOL_VERSION,
        },
        'outputs/results/temperature.pt',
    )
    try:
        plot_reliability_diagram(
            probs, labels, n_bins=15,
            save_path=os.path.join(settings.paths.figures, 'reliability_tier2_eff.png'),
        )
    except Exception as _e:
        print('   reliability diagram skipped:', _e)
    print(f'   ✅ Validation-Split temperature={float(temperature):.4f}  ece={float(ece):.4f} (backbone={backbone})')

produce(
    'calibration_temperature',
    outputs=['outputs/results/temperature.pt'],
    fn=_calibrate_temperature,
    inputs=['outputs/models/best_tier2_efficientnet.pth', 'data/processed/val.csv'],
)


## 10. Split-Conformal Calibration (Disjoint Calibration Split -> q_hat.pt)

In [ ]:
# ----------------------------------------------------------------------------
# Split-conformal quantile (q_hat) calibration runs STRICTLY AFTER Validation
# Split temperature scaling. Consumes temperature.pt, verifies zero patient
# overlap between val.csv and calibration.csv, computes non-conformity scores
# exclusively on data/processed/calibration.csv using temperature-scaled
# probabilities, and logs marginal/class-conditional coverage + set-size rates.
# ----------------------------------------------------------------------------
def _calibrate_conformal():
    import json
    import pandas as pd
    import torch
    from torch.utils.data import DataLoader
    from tcrp.config import get_settings
    from tcrp.data.augmentation import ClassicalAugmentation
    from tcrp.data.datasets import NIHChestXrayDataset
    from tcrp.data.splits import verify_patient_disjointness
    from tcrp.models import ModelFactory
    from tcrp.uncertainty.conformal import ConformalPredictor

    val_df = pd.read_csv('data/processed/val.csv')
    cal_df = pd.read_csv('data/processed/calibration.csv')
    verify_patient_disjointness({'val': val_df, 'calibration': cal_df})

    temp_state = torch.load('outputs/results/temperature.pt', map_location='cpu', weights_only=False)
    if not isinstance(temp_state, dict) or 'temperature' not in temp_state:
        raise RuntimeError('outputs/results/temperature.pt is missing Validation-Split temperature scalar')
    if temp_state.get('split', 'val') != 'val':
        raise RuntimeError(f'Temperature must be fitted on val split, got {temp_state.get("split")!r}')
    temperature = float(temp_state['temperature'])
    backbone = str(temp_state.get('backbone', 'efficientnet_b4'))
    weights_path = str(temp_state.get('weights_path', 'outputs/models/best_tier2_efficientnet.pth'))
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f'Tier 2 weights not found for conformal calibration: {weights_path}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    settings = get_settings()
    model = ModelFactory.create(backbone, num_classes=2, pretrained=False).to(device)
    state = torch.load(weights_path, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state_dict', state) if isinstance(state, dict) else state)
    model.eval()

    eval_tf = ClassicalAugmentation(image_size=settings.data.image_size, is_training=False).pipeline
    calib_ds = NIHChestXrayDataset(csv_file='data/processed/calibration.csv', transform=eval_tf)
    calib_loader = DataLoader(calib_ds, batch_size=32, shuffle=False, num_workers=0)

    cp = ConformalPredictor(alpha=1.0 - settings.evaluation.conformal_coverage)
    q_hat = cp.calibrate(model, calib_loader, device, temperature=temperature)
    os.makedirs('outputs/results', exist_ok=True)
    torch.save(
        {
            'q_hat': float(q_hat),
            'alpha': float(cp.alpha),
            'temperature': float(temperature),
            'backbone': backbone,
            'temperature_split': 'val',
            'conformal_split': 'calibration',
            'protocol': PROTOCOL_VERSION,
        },
        'outputs/results/q_hat.pt',
    )
    report = cp.last_calibration_report
    summary = {
        'protocol': PROTOCOL_VERSION,
        'backbone': backbone,
        'temperature_split': 'val',
        'conformal_split': 'calibration',
        'temperature_before_conformal': True,
        **(report.to_dict() if report is not None else {'q_hat': float(q_hat), 'temperature': float(temperature)}),
    }
    with open('outputs/results/conformal_calibration_summary.json', 'w') as _cf:
        json.dump(summary, _cf, indent=2)
    print(
        f'   ✅ Split-conformal calibration complete: q_hat={q_hat:.4f} (T_cal={temperature:.4f}, '
        f'marginal_cov={summary.get("marginal_coverage", 0.0):.4f}, '
        f'singleton={summary.get("singleton_rate", 0.0):.4f}, '
        f'two_class={summary.get("two_class_abstention_rate", 0.0):.4f}, '
        f'empty_fallback={summary.get("empty_fallback_rate", 0.0):.4f})'
    )

produce(
    'calibration_conformal',
    outputs=['outputs/results/q_hat.pt', 'outputs/results/conformal_calibration_summary.json'],
    fn=_calibrate_conformal,
    inputs=['outputs/results/temperature.pt', 'data/processed/val.csv', 'data/processed/calibration.csv'],
)


## 11. CheXpert Zero-Shot Test Set (CONDITIONAL — downloads only if A14 will run)

In [ ]:
# ----------------------------------------------------------------------------
# Conditional CheXpert zero-shot test set: download ONLY if A14 will run.
# ----------------------------------------------------------------------------
import glob, os, shutil, subprocess, sys
from pathlib import Path

if not any(_will_run(s) for s in CHEXPERT_CONSUMERS):
    print('⏭️ CheXpert download SKIPPED — A14 zero-shot eval not enabled or already done.')
else:
    os.chdir(PROJECT_ROOT)
    CHX_KAGGLE  = 'ashery/chexpert'
    COLAB_CHX   = '/content/chexpert_data'
    DRIVE_CHX   = '/content/drive/MyDrive/chexpert_valid'
    CHX_CSV_OUT = os.path.join(PROJECT_ROOT, 'data', 'processed', 'chexpert_test.csv')
    KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'
    os.makedirs(os.path.dirname(CHX_CSV_OUT), exist_ok=True)

    def _find(root, name):
        hits = glob.glob(os.path.join(root, '**', name), recursive=True)
        return hits[0] if hits else None

    def _find_valid_dir(root):
        for d in glob.glob(os.path.join(root, '**', 'valid'), recursive=True):
            if os.path.isdir(d) and glob.glob(os.path.join(d, 'patient*')):
                return d
        return None

    if not _find_valid_dir(COLAB_CHX) and os.path.isdir(DRIVE_CHX):
        print('📂 Restoring cached CheXpert valid split from Drive...')
        os.makedirs(COLAB_CHX, exist_ok=True)
        subprocess.check_call(['cp', '-r', f'{DRIVE_CHX}/.', COLAB_CHX])

    valid_dir = _find_valid_dir(COLAB_CHX)
    valid_csv = _find(COLAB_CHX, 'valid.csv')
    if not valid_dir or not valid_csv:
        if not KAGGLE_JSON.exists():
            print('⚠️ kaggle.json required for CheXpert (run the credentials cell). Skipping A14 data.')
        else:
            os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_JSON.parent)
            print(f'🌐 Downloading {CHX_KAGGLE} (one-time, several minutes)...')
            os.makedirs(COLAB_CHX, exist_ok=True)
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d', CHX_KAGGLE, '-p', COLAB_CHX, '--unzip'])
            valid_dir = _find_valid_dir(COLAB_CHX)
            valid_csv = _find(COLAB_CHX, 'valid.csv')
            if valid_dir and os.path.isdir('/content/drive/MyDrive'):
                print('💾 Caching valid/ + valid.csv to Drive...')
                os.makedirs(DRIVE_CHX, exist_ok=True)
                subprocess.check_call(['cp', '-r', valid_dir, DRIVE_CHX])
                if valid_csv:
                    shutil.copy2(valid_csv, os.path.join(DRIVE_CHX, 'valid.csv'))
            # Sample a larger patient-disjoint zero-shot cohort (up to 4,000 frontal images,
            # excluding uncertain -1 labels) from CheXpert train.csv BEFORE cleaning up train/.
            train_csv = _find(COLAB_CHX, 'train.csv')
            train_dirs = [d for d in glob.glob(os.path.join(COLAB_CHX, '**', 'train'), recursive=True) if os.path.isdir(d)]
            CHX_LARGE_OUT = os.path.join(PROJECT_ROOT, 'data', 'processed', 'chexpert_large_test.csv')
            CHX_LARGE_IMG = os.path.join(COLAB_CHX, 'large_sample')
            if train_csv and train_dirs:
                os.makedirs(CHX_LARGE_IMG, exist_ok=True)
                sample_code = r'''
import os, shutil, pandas as pd
tcsv = os.environ['CHX_TRAIN_CSV']; tdir = os.environ['CHX_TRAIN_DIR']
out_csv = os.environ['CHX_LARGE_OUT']; dst_dir = os.environ['CHX_LARGE_IMG']
df = pd.read_csv(tcsv)
if 'Frontal/Lateral' in df.columns:
    df = df[df['Frontal/Lateral'] == 'Frontal'].copy()
# Exclude uncertain (-1.0) Pneumothorax labels; keep definitive positive (1.0) or definitive No Finding / negative (0.0)
p_col = df['Pneumothorax']
nf_col = df['No Finding'] if 'No Finding' in df.columns else pd.Series(0, index=df.index)
pos = df[p_col == 1.0].copy()
neg = df[(p_col == 0.0) | ((p_col.isna()) & (nf_col == 1.0))].copy()
pos['patient'] = pos['Path'].astype(str).str.extract(r'(patient\d+)')[0]
neg['patient'] = neg['Path'].astype(str).str.extract(r'(patient\d+)')[0]
neg = neg[~neg['patient'].isin(set(pos['patient']))].copy()
pos_s = pos.drop_duplicates('patient').sample(n=min(800, pos['patient'].nunique()), random_state=42)
neg_s = neg.drop_duplicates('patient').sample(n=min(3200, neg['patient'].nunique()), random_state=42)
samp = pd.concat([pos_s, neg_s], ignore_index=True).drop(columns=['patient'])
kept = []
for idx, row in samp.iterrows():
    p = str(row['Path'])
    sub = p.split('train/', 1)[1] if 'train/' in p else os.path.basename(p)
    src = os.path.join(tdir, sub)
    if os.path.exists(src):
        dst = os.path.join(dst_dir, sub)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        row_copy = row.copy()
        row_copy['Path'] = dst
        kept.append(row_copy)
pd.DataFrame(kept).to_csv(out_csv, index=False)
print(f'CHEXPERT_LARGE_ROWS={len(kept)}')
'''
                env_l = dict(os.environ, CHX_TRAIN_CSV=train_csv, CHX_TRAIN_DIR=train_dirs[0],
                             CHX_LARGE_OUT=CHX_LARGE_OUT, CHX_LARGE_IMG=CHX_LARGE_IMG)
                subprocess.check_call([sys.executable, '-c', sample_code], env=env_l)
            for train_dir in train_dirs:
                shutil.rmtree(train_dir, ignore_errors=True)

    if valid_dir and valid_csv:
        print(f'✅ valid images: {valid_dir}')
        build_code = r"""
import os, pandas as pd
valid_dir = os.environ['CHX_VALID_DIR']; valid_csv = os.environ['CHX_VALID_CSV']; out = os.environ['CHX_OUT']
df = pd.read_csv(valid_csv)
def to_abs(p):
    p = str(p)
    sub = p.split('valid/', 1)[1] if 'valid/' in p else os.path.basename(p)
    return os.path.join(valid_dir, sub)
df['Path'] = df['Path'].apply(to_abs)
if 'Frontal/Lateral' in df.columns:
    df = df[df['Frontal/Lateral'] == 'Frontal'].copy()
df = df[df['Path'].apply(os.path.exists)].copy()
df.to_csv(out, index=False)
print('CHEXPERT ROWS=%d' % len(df))
"""
        env = dict(os.environ, CHX_VALID_DIR=valid_dir, CHX_VALID_CSV=valid_csv, CHX_OUT=CHX_CSV_OUT)
        subprocess.check_call([sys.executable, '-c', build_code], env=env)
        if os.path.isdir('/content/drive/MyDrive'):
            os.makedirs(DRIVE_CHX, exist_ok=True)
            shutil.copy2(CHX_CSV_OUT, os.path.join(DRIVE_CHX, 'chexpert_test.csv'))
        print(f'✅ Wrote {CHX_CSV_OUT} (A14 will evaluate real CheXpert frontals).')
    else:
        print('⚠️ CheXpert valid split unavailable — A14 will be BLOCKED (no chexpert_test.csv).')


## 12. Evaluation Ablations A13 / A14 (if missing)

In [ ]:
# ----------------------------------------------------------------------------
# Evaluation ablations A13 (tiered, NIH) and A14 (CheXpert zero-shot).
# Only run if their result JSON is missing — the trained weights are reused.
# ----------------------------------------------------------------------------
def _eval_a13():
    sh([sys.executable, '-u', 'scripts/evaluate_tiered.py',
        '--run-name', 'A13_Tiered_ArkPlus',
        '--tier2-backbone', 'ark_plus', '--dynamic-threshold', 'true'])

produce('eval_A13_tiered',
        outputs=['outputs/results/A13_Tiered_ArkPlus.json'],
        fn=_eval_a13,
        inputs=['outputs/models/best_tier1_mobilenet.pth', 'data/processed/test.csv'])

def _eval_a14():
    sh([sys.executable, '-u', 'scripts/evaluate_chexpert.py',
        '--run-name', 'A14_CheXpert_ZeroShot', '--tier2-backbone', 'ark_plus'])

produce('eval_A14_chexpert',
        outputs=['outputs/results/A14_CheXpert_ZeroShot.json'],
        fn=_eval_a14,
        inputs=['data/processed/chexpert_test.csv'])

def _eval_a14b():
    sh([sys.executable, '-u', 'scripts/evaluate_chexpert.py',
        '--run-name', 'A14b_CheXpert_Large_ZeroShot', '--tier2-backbone', 'ark_plus',
        '--csv', 'data/processed/chexpert_large_test.csv'])

produce('eval_A14b_chexpert_large',
        outputs=['outputs/results/A14b_CheXpert_Large_ZeroShot.json'],
        fn=_eval_a14b,
        inputs=['data/processed/chexpert_large_test.csv'])


# ----------------------------------------------------------------------------
# Remaining ablation rows A1-A12 + A15 (full PLAN.md table). Each loads its REAL
# trained weights and writes outputs/results/<run_name>.json via the unified
# scripts/evaluate_ablation.py. Skip-existing + input-gating keep this dynamic:
# a row is BLOCKED until its weights exist, SKIPPED once its result JSON exists.
# ----------------------------------------------------------------------------
_T1  = 'outputs/models/best_tier1_mobilenet.pth'
_EFF = 'outputs/models/best_tier2_efficientnet.pth'
# Extra weight inputs (beyond test.csv) that gate each evaluation row.
_EVAL_WEIGHTS = {
    'A1':  [_T1],
    'A2':  [_EFF],
    'A3':  [_T1, _EFF], 'A4': [_T1, _EFF], 'A5': [_T1, _EFF], 'A5b': [_T1, _EFF],
    'A6':  [_T1, _EFF], 'A7': [_T1, _EFF],
    'A8':  [_T1, 'outputs/models/A8_NoSynthetic/best_model.pth'],
    'A9':  [_T1, 'outputs/models/A9_NoAugmentation/best_model.pth'],
    'A10': [_EFF],
    'A11': ['outputs/models/A11_ArkPlus_Only_NoMCTTA/best_model.pth'],
    'A12': ['outputs/models/A12_ArkPlus_Only_MC_TTA/best_model.pth'],
    'A15': [_T1, 'outputs/models/A15_Mixup_Cutmix/best_model.pth'],
}

def _make_eval_ablation(eid):
    def _run():
        cmd = [sys.executable, '-u', 'scripts/evaluate_ablation.py', '--ablation', eid,
               '--batch-size', str(EVAL_HP['batch_size']),
               '--micro-batch', str(EVAL_HP['micro_batch']),
               '--num-workers', str(EVAL_HP['num_workers'])]
        cmd += ['--amp'] if EVAL_HP['amp'] else ['--no-amp']
        if EVAL_HP['mc_passes'] is not None:
            cmd += ['--mc-passes', str(EVAL_HP['mc_passes'])]
        if EVAL_HP['tta_passes'] is not None:
            cmd += ['--tta-passes', str(EVAL_HP['tta_passes'])]
        sh(cmd)
    return _run

for _step, (_eid, _rn) in EVAL_ABLATIONS.items():
    produce(_step,
            outputs=[f'outputs/results/{_rn}.json'],
            fn=_make_eval_ablation(_eid),
            inputs=['data/processed/test.csv'] + _EVAL_WEIGHTS[_eid])


## 13. KEYSTONE: Per-Image Predictions (Ark+ and EfficientNet)

In [ ]:
# ----------------------------------------------------------------------------
# KEYSTONE: real per-image predictions for BOTH Tier-2 backbones.
# tiered_predictions.csv feeds the 5 analysis notebooks + statistical tests.
# ----------------------------------------------------------------------------
def _pred_ark():
    sh([sys.executable, '-u', 'scripts/export_predictions.py',
        '--tier2-backbone', 'ark_plus', '--mc-passes', str(MC_PASSES),
        '--output', 'outputs/results/tiered_predictions.csv'])

produce('predictions_arkplus',
        outputs=['outputs/results/tiered_predictions.csv'],
        fn=_pred_ark,
        inputs=['outputs/models/best_tier1_mobilenet.pth', 'data/processed/test.csv'])

def _pred_eff():
    sh([sys.executable, '-u', 'scripts/export_predictions.py',
        '--tier2-backbone', 'efficientnet_b4', '--mc-passes', str(MC_PASSES),
        '--output', 'outputs/results/tiered_predictions_efficientnet.csv',
        '--val-output', 'outputs/results/val_predictions_efficientnet.csv'])

produce('predictions_efficientnet',
        outputs=['outputs/results/tiered_predictions_efficientnet.csv'],
        fn=_pred_eff,
        inputs=['outputs/models/best_tier2_efficientnet.pth', 'data/processed/test.csv'])


## 14. Figure-Input CSVs (derived from the real predictions)

In [ ]:
# ----------------------------------------------------------------------------
# Figure-input CSVs derived purely from the REAL predictions (no images needed):
#   threshold_sweep.csv, tier_transition_log.csv
# Validation-Split predictions stay untouched at outputs/results/val_predictions.csv.
# These feed scripts/generate_report_figures.py.
# ----------------------------------------------------------------------------
def _figure_inputs():
    import numpy as np, pandas as pd
    df = pd.read_csv('outputs/results/tiered_predictions.csv')
    os.makedirs('outputs/results', exist_ok=True)
    t1 = df['tier1_prob'].to_numpy(); t2 = df['tier2_prob'].to_numpy(); y = df['y_true'].to_numpy()
    conf1 = np.maximum(t1, 1.0 - t1)

    rows = []
    for theta in np.round(np.arange(0.50, 0.96, 0.05), 2):
        esc = conf1 < theta
        prob = np.where(esc, t2, t1)
        pred = (prob >= 0.5).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
        fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
        n = max(len(y), 1)
        acc = (tp + tn) / n
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        pct2 = 100.0 * float(esc.mean())
        # Relative throughput proxy: Tier 2 ~6x slower than Tier 1.
        fps = 1000.0 / (5.0 + (pct2 / 100.0) * 25.0)
        rows.append(dict(theta=float(theta), accuracy=acc, sensitivity=sens,
                         specificity=spec, percent_tier2=pct2, estimated_fps=fps))
    pd.DataFrame(rows).to_csv('outputs/results/threshold_sweep.csv', index=False)

    pd.DataFrame(dict(confidence=df['tiered_prob'], tier_used=df['tier_used'],
                      ground_truth=df['y_true'])).to_csv(
        'outputs/results/tier_transition_log.csv', index=False)
    print('   wrote threshold_sweep.csv, tier_transition_log.csv')

produce('figure_input_csvs',
        outputs=['outputs/results/threshold_sweep.csv',
                 'outputs/results/tier_transition_log.csv'],
        fn=_figure_inputs,
        inputs=['outputs/results/tiered_predictions.csv'],
        always=True)


## 15. ablation.json + Statistical Tests + Model Comparison

In [ ]:
# ----------------------------------------------------------------------------
# ablation.json (real metrics) + statistical tests + paired model comparison.
# These are cheap derivations → always=True so they reflect the latest data.
# ----------------------------------------------------------------------------
def _ablation_json():
    sh([sys.executable, '-u', 'scripts/build_ablation_json.py'])

produce('ablation_json', outputs=['outputs/results/ablation.json'],
        fn=_ablation_json, always=True)

def _stats():
    sh([sys.executable, '-u', 'scripts/statistical_tests.py', '--n-iterations', str(N_BOOT)])

produce('statistical_tests', outputs=['outputs/results/statistical_comparison.csv'],
        fn=_stats, inputs=['outputs/results/tiered_predictions.csv'], always=True)

def _compare():
    sh([sys.executable, '-u', 'scripts/compare_models.py', '--n-iterations', str(N_BOOT)])

produce('compare_models', outputs=['outputs/results/model_comparison_bootstrap.csv'],
        fn=_compare, inputs=['outputs/results/tiered_predictions.csv'], always=True)


def _revision_experiments():
    sh([sys.executable, '-u', 'scripts/revision_experiments.py', '--all', '--n-permutations', '10'])
    os.makedirs('outputs/provenance', exist_ok=True)
    with open('outputs/provenance/requirements-lock.txt', 'w') as _rf:
        subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=_rf, check=True)

produce('revision_experiments',
        outputs=['outputs/results/order_sensitivity.json',
                 'outputs/results/clinical_threshold_sweep.csv',
                 'outputs/results/calibration_summary.json',
                 'outputs/results/dataset_audit.json'],
        fn=_revision_experiments,
        inputs=['outputs/results/tiered_predictions.csv'],
        always=True)


## 16. Thesis Figures + 5 Analysis Notebooks (headless)

In [ ]:
# ----------------------------------------------------------------------------
# Thesis figures (generate_report_figures.py) + the 5 analysis notebooks
# executed headless via nbconvert (real predictions only).
# ----------------------------------------------------------------------------
def _report_figs():
    sh([sys.executable, '-u', 'scripts/generate_report_figures.py'])

produce('report_figures', outputs=['thesis/figures/threshold_sweep_4panel.png'],
        fn=_report_figs, inputs=['outputs/results/threshold_sweep.csv'], always=True)

os.makedirs('outputs/notebooks', exist_ok=True)
# NB_STEPS comes from the config cell (single source of truth).

def _make_runner(stem):
    def _run():
        # PYTHONPATH includes src/ and repo root so analysis notebooks can `import tcrp...`
        # even though nbconvert runs them with cwd set to the notebooks/ directory.
        sh([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
            '--ExecutePreprocessor.timeout=1800',
            '--output-dir', 'outputs/notebooks', '--output', stem,
            f'notebooks/{stem}.ipynb'],
           env={'PYTHONPATH': os.path.abspath('src') + os.pathsep + os.path.abspath('.')})
    return _run

for _step, _stem in NB_STEPS.items():
    # synthetic_quality needs generated SD images; without them it is BLOCKED, not an error.
    _inp = (['data/synthetic/synthetic_metadata.csv'] if _stem == 'synthetic_quality'
            else ['outputs/results/tiered_predictions.csv'])
    produce(_step, outputs=[f'outputs/notebooks/{_stem}.ipynb'],
            fn=_make_runner(_stem), inputs=_inp)


## 17. ONNX Export + Latency/Carbon Benchmark

In [ ]:
# ----------------------------------------------------------------------------
# ONNX export (Tier 1 + Tier 2) and latency / carbon benchmark.
# ----------------------------------------------------------------------------
# torch 2.7's torch.onnx.export imports onnxscript; the base requirements install
# can miss it (resolver fallback). Ensure it on-demand here (idempotent, fast).
import subprocess as _sp
if _sp.run([sys.executable, '-c', 'import onnxscript'], capture_output=True).returncode != 0:
    print('🔧 Installing onnxscript for ONNX export...')
    _sp.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxscript'])

def _onnx1():
    sh([sys.executable, '-u', 'scripts/export_onnx.py', '--model', 'tier1'])

produce('onnx_tier1', outputs=['outputs/models/tier1_mobilenet.onnx'],
        fn=_onnx1, inputs=['outputs/models/best_tier1_mobilenet.pth'])

def _onnx2():
    sh([sys.executable, '-u', 'scripts/export_onnx.py', '--model', 'tier2'])

produce('onnx_tier2', outputs=['outputs/models/tier2_efficientnet.onnx'],
        fn=_onnx2, inputs=['outputs/models/best_tier2_efficientnet.pth'])

def _bench():
    os.makedirs('outputs/results', exist_ok=True)
    r = subprocess.run([sys.executable, '-u', 'scripts/benchmark_latency.py'],
                       capture_output=True, text=True)
    out = (r.stdout or '') + (('\n' + r.stderr) if r.stderr else '')
    if r.returncode != 0:
        print(out[-1200:])
        raise RuntimeError(f'benchmark_latency failed (exit {r.returncode})')
    with open('outputs/results/benchmark_latency.txt', 'w') as f:
        f.write(out)
    print(out[-800:])

produce('benchmark_latency', outputs=['outputs/results/benchmark_latency.txt'], fn=_bench)


## 18.5 Scientific Integrity + Provenance Gate

In [ ]:
# Final gate: a green production report is not enough; prove this protocol's
# splits, checkpoints, predictions, and result JSONs belong to the clean rerun.
def _integrity_gate():
    import json
    import pandas as pd, torch
    from tcrp.evaluation.provenance import build_provenance_manifest, verify_provenance_manifest

    _split_manifest = 'outputs/provenance/split_manifest.json'
    if not os.path.exists(_split_manifest):
        raise RuntimeError('missing patient-disjoint split manifest')
    with open(_split_manifest) as _f:
        _sm = json.load(_f)
    if _sm.get('protocol') != PROTOCOL_VERSION:
        raise RuntimeError(f'wrong split protocol: {_sm.get("protocol")}')

    _critical_steps = (
        ['synthetic_generation', 'train_tier1', 'train_tier2_efficientnet', 'train_tier2_arkplus']
        + list(ABLATIONS)
        + ['calibration_temperature', 'calibration_conformal']
        + list(EVAL_ABLATIONS)
        + ['eval_A13_tiered', 'eval_A14_chexpert',
           'predictions_arkplus', 'predictions_efficientnet', 'revision_experiments']
    )
    if PATIENT_DISJOINT_REBUILD:
        _not_fresh = [s for s in _critical_steps
                      if os.path.abspath(MARKERS[s]) not in PRODUCED]
        if _not_fresh:
            raise RuntimeError(f'clean rebuild incomplete; not produced this session: {_not_fresh}')

    _test_n = _sm['splits']['test']['rows']
    for _pred in ['outputs/results/tiered_predictions.csv',
                  'outputs/results/tiered_predictions_efficientnet.csv']:
        _n = len(pd.read_csv(_pred))
        if _n != _test_n:
            raise RuntimeError(f'{_pred} rows={_n}, expected patient-disjoint test rows={_test_n}')

    import math
    _temp_state = torch.load('outputs/results/temperature.pt', map_location='cpu', weights_only=False)
    _qhat_state = torch.load('outputs/results/q_hat.pt', map_location='cpu', weights_only=False)
    with open('outputs/results/conformal_calibration_summary.json') as _cf:
        _cp_summary = json.load(_cf)
    if not isinstance(_temp_state, dict) or not isinstance(_qhat_state, dict):
        raise RuntimeError('temperature.pt and q_hat.pt must be structured metadata dicts')
    _t_split = _temp_state.get('split')
    _q_t_split = _qhat_state.get('temperature_split')
    _q_c_split = _qhat_state.get('conformal_split')
    if _t_split != 'val' or _q_t_split != 'val' or _q_c_split != 'calibration':
        raise RuntimeError(
            f'calibration split provenance mismatch: temp.split={_t_split!r}, '
            f'qhat.temperature_split={_q_t_split!r}, qhat.conformal_split={_q_c_split!r}'
        )
    if _temp_state.get('backbone') != _qhat_state.get('backbone') or not _temp_state.get('backbone'):
        raise RuntimeError(
            f'calibration backbone mismatch: temp={_temp_state.get("backbone")!r}, '
            f'qhat={_qhat_state.get("backbone")!r}'
        )
    _t_val = float(_temp_state['temperature'])
    _q_t_val = float(_qhat_state['temperature'])
    if not math.isclose(_t_val, _q_t_val, rel_tol=1e-3, abs_tol=1e-4):
        raise RuntimeError(
            f'temperature.pt ({_t_val:.4f}) and q_hat.pt ({_q_t_val:.4f}) temperature mismatch'
        )
    _cal_meta = {
        'backbone': str(_temp_state['backbone']),
        'temperature_split': str(_t_split),
        'conformal_split': str(_q_c_split),
        'temperature_before_conformal': _cp_summary.get('temperature_before_conformal') is True,
        'temperature': _t_val,
        'q_hat': float(_qhat_state['q_hat']),
    }

    _artifact_paths = sorted(
        {MARKERS[s] for s in _critical_steps if s in MARKERS}
        | {
            'data/synthetic/fid_log.csv',
            'outputs/results/conformal_calibration_summary.json',
            'outputs/results/ablation.json',
            'outputs/results/statistical_comparison.csv',
            'outputs/results/calibration_summary.json',
        }
    )
    _manifest_out = 'outputs/provenance/rerun_manifest.json'
    build_provenance_manifest(
        output_path=_manifest_out,
        artifact_paths=_artifact_paths,
        protocol=PROTOCOL_VERSION,
        split_manifest_path=_split_manifest,
        calibration_metadata=_cal_meta,
        base_dir=PROJECT_ROOT,
    )
    _ver = verify_provenance_manifest(
        manifest_path=_manifest_out,
        base_dir=PROJECT_ROOT,
        expected_protocol=PROTOCOL_VERSION,
        required_artifacts=['outputs/results/temperature.pt', 'outputs/results/q_hat.pt'],
        strict=True,
    )
    print(f'✅ SCIENTIFIC INTEGRITY GATE PASSED: {PROTOCOL_VERSION} ({_ver.verified_artifacts} artifacts verified)')

produce('integrity_gate', outputs=['outputs/provenance/rerun_manifest.json'],
        fn=_integrity_gate, always=True)


## 19. Production Report

In [ ]:
# ----------------------------------------------------------------------------
# Production report: what ran, what was skipped, what failed.
# ----------------------------------------------------------------------------
from collections import Counter
print('=' * 64)
print('📋 PRODUCTION REPORT')
print('=' * 64)
for step, status, detail in RUN_REPORT:
    icon = {'DONE':'✅','SKIPPED':'⏭️','DISABLED':'🚫','BLOCKED':'⛔','PARTIAL':'⚠️','ERROR':'❌'}.get(status, '•')
    print(f'  {icon} {status:<9} {step:<28} {detail}')
print('-' * 64)
cnt = Counter(s for _, s, _ in RUN_REPORT)
print('  ' + '   '.join(f'{k}={v}' for k, v in cnt.items()))
print(f'  Files produced in this session: {len(PRODUCED)}')
_err = [s for s, st, _ in RUN_REPORT if st in ('ERROR', 'PARTIAL')]
if _err:
    print(f'  ⚠️ Review: {_err}  (errors are non-fatal; fix and re-run, completed steps are skipped)')
else:
    print('  🎉 No errors.')


## 20. Save to Drive — MERGE into a Single Tree (no base-model/zip duplication)

In [ ]:
# ----------------------------------------------------------------------------
# Save to Drive: MERGEs ONLY the files produced / new in THIS session into the
# single xray_outputs/outputs tree. Base model weights (large .pth) are NOT
# re-uploaded; no dated new zip copy is produced.
# ----------------------------------------------------------------------------
def _save_to_drive():
    if not SAVE_TO_DRIVE:
        print('SAVE_TO_DRIVE=False → save skipped.'); return
    if not (USE_DRIVE and os.path.isdir('/content/drive/MyDrive')):
        print('Drive not mounted → save skipped.'); return

    big = BIG_FILE_MB * 1024 * 1024
    pushed = skipped_big = skipped_same = 0
    for _rel in SYNC_DIRS:
        src_root = os.path.join(PROJECT_ROOT, _rel)
        if not os.path.isdir(src_root):
            continue
        dst_root = os.path.join(DRIVE_OUTPUT_DIR, _rel)
        os.makedirs(dst_root, exist_ok=True)
        for root, _, files in os.walk(src_root):
            for fn in files:
                sp = os.path.join(root, fn)
                rel = os.path.relpath(sp, src_root)
                dp = os.path.join(dst_root, rel)
                produced = os.path.abspath(sp) in PRODUCED
                if not produced:
                    try:
                        if os.path.getsize(sp) > big:   # base weight not regenerated now — never re-upload
                            skipped_big += 1; continue
                    except OSError:
                        continue
                    if os.path.exists(dp):              # unchanged small file already on Drive
                        skipped_same += 1; continue
                os.makedirs(os.path.dirname(dp), exist_ok=True)
                try:
                    shutil.copy2(sp, dp); pushed += 1   # produced -> overwrites the Drive copy
                except Exception as e:
                    print(f'   ⚠️ copy failed {_rel}/{rel}: {e}')

    print(f'✅ Merge complete → {DRIVE_OUTPUT_DIR}  (dirs: {SYNC_DIRS})')
    print(f'   pushed={pushed}  skipped_base_weights={skipped_big}  skipped_unchanged={skipped_same}')

    if MAKE_ZIP_SNAPSHOT:
        import zipfile
        # Build the zip on LOCAL disk first — writing a big zip straight to the Drive
        # FUSE mount is unreliable — then copy it over. Excludes the multi-GB base
        # weights (they live in the Drive tree); includes produced results/figures/notebooks.
        local_zip = '/content/xray_results_snapshot.zip'
        n_zipped = 0
        with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as z:
            for _rel in SYNC_DIRS:
                base = os.path.join(PROJECT_ROOT, _rel)
                if not os.path.isdir(base):
                    continue
                for root, _, files in os.walk(base):
                    for fn in files:
                        ap = os.path.join(root, fn)
                        try:
                            if os.path.getsize(ap) > big and os.path.abspath(ap) not in PRODUCED:
                                continue
                        except OSError:
                            continue
                        z.write(ap, os.path.join(_rel, os.path.relpath(ap, base)))
                        n_zipped += 1
        drive_zip = os.path.join(DRIVE_OUTPUT_DIR, 'xray_results_snapshot.zip')  # SAME path, overwritten
        shutil.copy2(local_zip, drive_zip)
        print(f'   📦 zip: {n_zipped} files, {os.path.getsize(local_zip) / 1e6:.1f} MB -> {drive_zip}')

    # CRITICAL: Drive (FUSE) writes are buffered/async — files & the zip can appear
    # "missing" on Drive if the session ends before they sync. Flush now so everything
    # is durably persisted to Google Drive before this cell returns.
    try:
        from google.colab import drive as _gdrive
        print('💾 Flushing writes to Google Drive (this makes them durable)...')
        _gdrive.flush_and_unmount()
        print('✅ Drive flushed & unmounted — all files + zip are now on Google Drive.')
        print('   (Re-run the "Mount Google Drive" cell if you want to keep using Drive.)')
    except Exception as _fe:
        print(f'⚠️ Could not flush/unmount Drive ({_fe}); allow a minute for Drive to sync.')

_save_to_drive()
